# Audite as viagens antes de preparar

Uma equipe quer prever a duração de uma viagem antes de ela partir. Cada linha é uma viagem fictícia. O contrato da entrada prevê ID textual, data ISO, código de estação, distância planejada em km, paradas como contagem, duração real em minutos e faturamento em minutos após a chegada. O segundo CSV relaciona códigos e nomes de estações. Comece com os arquivos sintéticos da atividade. Preserve-os. Revise [duplicatas e qualidade](https://devrgs.github.io/Aprendendo-linguagens/#/python/exploracao/qualidade-dos-dados).

Compare duplicata exata com conflito de ID. Retire apenas a cópia idêntica. Se houver dados divergentes com mesmo ID, os critérios do projeto exigem interromper para esclarecimento. Rejeite distâncias negativas e alvos ausentes ou inválidos. Ausência de distância pode ser tratada depois. Zero paradas é válido. Guarde o ID em todas as decisões e use uma junção à esquerda para encontrar estações sem cadastro.

Execute as células na ordem com **Shift+Enter**. A preparação cria uma pasta nova na área de arquivos do caderno. Baixe o caderno editado e os resultados que quiser guardar.


In [ ]:
from pathlib import Path as _Path
from tempfile import mkdtemp as _mkdtemp
import os as _os
import sys as _sys
from importlib import invalidate_caches as _refresh_imports

# Auxiliares editáveis devem ser relidos sem aproveitar bytecode antigo.
_sys.dont_write_bytecode = True

_raiz = _Path.cwd()
_trabalho = _Path(_mkdtemp(prefix='python-auditar-', dir=_raiz))
import numpy
import pandas
import matplotlib


## E1 · Preveja e execute

Leia os dois CSVs e conte observações antes de limpar.


In [ ]:
_pasta = _trabalho / 'e1'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'dados/estacoes.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('estacao,nome\nA,Centro\nB,Norte\n', encoding='utf-8')
_arquivo = _pasta / 'dados/viagens.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\nV11,2026-01-11,A,-1,1,7,7\nV04,2026-01-04,B,6,2,10,10\n', encoding='utf-8')
_refresh_imports()


In [ ]:
import pandas as pd
viagens = pd.read_csv("dados/viagens.csv", dtype={"id":"string"})
estacoes = pd.read_csv("dados/estacoes.csv", dtype="string")
print(viagens.shape)
print(estacoes["estacao"].tolist())
print(viagens["id"].iloc[0])


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
(12, 7)
['A', 'B']
V01
```

O cabeçalho descreve sete colunas. A tabela de estações tem duas chaves e a de viagens tem 12 registros brutos.


## E2 · Altere ou depure

Localize a cópia extra, as distâncias ausentes e os zeros de paradas, sem editar os dados.


In [ ]:
_pasta = _trabalho / 'e2'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'dados/estacoes.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('estacao,nome\nA,Centro\nB,Norte\n', encoding='utf-8')
_arquivo = _pasta / 'dados/viagens.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\nV11,2026-01-11,A,-1,1,7,7\nV04,2026-01-04,B,6,2,10,10\n', encoding='utf-8')
_refresh_imports()


In [ ]:
import pandas as pd
brutas = pd.read_csv("dados/viagens.csv")
print(brutas.loc[brutas.duplicated(), "id"].tolist())
print(brutas.loc[brutas["km_planejados"].isna(), "id"].tolist())
print(brutas.loc[brutas["paradas_planejadas"].eq(0), "id"].tolist())


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
['V04']
['V03', 'V08']
['V01', 'V08']
```

Zero é uma observação. Ausente é uma lacuna. V08 mostra que uma linha pode ter os dois em colunas diferentes.


## E3 · Resolva

Produza auditoria.json com contagem antes e depois, motivos, unidades e IDs. Faça uma junção que preserve viagens e identifique a estação ausente.

Se editar um auxiliar no painel de arquivos, salve a edição. Execute novamente a preparação desta etapa e a célula do exercício. A preparação inicial do caderno cria outra pasta.


In [ ]:
_pasta = _trabalho / 'e3'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'dados/estacoes.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('estacao,nome\nA,Centro\nB,Norte\n', encoding='utf-8')
_arquivo = _pasta / 'dados/viagens.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\nV11,2026-01-11,A,-1,1,7,7\nV04,2026-01-04,B,6,2,10,10\n', encoding='utf-8')
_arquivo = _pasta / 'auditoria_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('import json\n\nimport numpy as np\nimport pandas as pd\n\nTIPOS = {\n    "id": "string",\n    "data": "string",\n    "estacao": "string",\n    "km_planejados": "float64",\n    "paradas_planejadas": "Int64",\n    "minutos_reais": "float64",\n    "minutos_faturados": "float64",\n}\n\nUNIDADES = {\n    "id": "chave de viagem",\n    "data": "dia (AAAA-MM-DD)",\n    "estacao": "chave de estação",\n    "km_planejados": "km",\n    "paradas_planejadas": "paradas (contagem)",\n    "minutos_reais": "min",\n    "minutos_faturados": "min",\n}\n\n\ndef ler_viagens(caminho):\n    viagens = pd.read_csv(caminho, dtype=TIPOS, keep_default_na=False, na_values=[""])\n    if set(viagens.columns) != set(TIPOS):\n        raise ValueError("colunas de viagens diferentes do esquema")\n    if viagens[["id", "data", "estacao"]].isna().any().any():\n        raise ValueError("id, data e estacao devem ser preenchidos")\n    if not viagens["data"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}").all():\n        raise ValueError("data deve estar no formato AAAA-MM-DD")\n    viagens["data"] = pd.to_datetime(viagens["data"], format="%Y-%m-%d")\n    return viagens\n\n\ndef ler_estacoes(caminho):\n    estacoes = pd.read_csv(caminho, dtype="string", keep_default_na=False, na_values=[""])\n    if set(estacoes.columns) != {"estacao", "nome"} or estacoes.isna().any().any():\n        raise ValueError("estacoes.csv deve ter chave e nome preenchidos")\n    return estacoes\n\n\ndef auditar(brutas, estacoes):\n    duplicatas = brutas.loc[brutas.duplicated(), "id"].value_counts().sort_index()\n    unicas = brutas.drop_duplicates()\n    conflitos = sorted(unicas.loc[unicas["id"].duplicated(keep=False), "id"].unique())\n    if conflitos:\n        prefixo = "ID em conflito: " if len(conflitos) == 1 else "IDs em conflito: "\n        raise ValueError(prefixo + ", ".join(conflitos))\n\n    regras = {\n        "km_planejados negativo": unicas["km_planejados"].lt(0),\n        "km_planejados infinito": np.isinf(unicas["km_planejados"]),\n        "paradas_planejadas ausente": unicas["paradas_planejadas"].isna(),\n        "paradas_planejadas negativo": unicas["paradas_planejadas"].lt(0).fillna(False),\n        "minutos_reais ausente": unicas["minutos_reais"].isna(),\n        "minutos_reais negativo": unicas["minutos_reais"].lt(0),\n        "minutos_reais infinito": np.isinf(unicas["minutos_reais"]),\n    }\n    rejeitadas = {}\n    for regra, mascara in regras.items():\n        for chave in sorted(unicas.loc[mascara, "id"]):\n            rejeitadas.setdefault(chave, []).append(regra)\n    validas = unicas.loc[~unicas["id"].isin(rejeitadas)].copy()\n    validas["paradas_planejadas"] = validas["paradas_planejadas"].astype("int64")\n    uniao = validas.merge(estacoes, on="estacao", how="left", validate="many_to_one", indicator=True)\n    auditoria = {\n        "linhas_brutas": len(brutas),\n        "linhas_validas": len(uniao),\n        "duplicatas_excedentes": {chave: int(n) for chave, n in duplicatas.items()},\n        "total_duplicatas_excedentes": int(duplicatas.sum()),\n        "rejeitadas": rejeitadas,\n        "total_rejeitadas": len(rejeitadas),\n        "km_ausentes_validas": sorted(validas.loc[validas["km_planejados"].isna(), "id"]),\n        "viagens_sem_cadastro": sorted(uniao.loc[uniao["_merge"] == "left_only", "id"]),\n        "decisoes": [\n            "Conservar o original e remover só cópias de linhas idênticas.",\n            "Interromper quando registros diferentes usam o mesmo id.",\n            "Conservar km ausente para imputação no preparo.",\n            "Conservar zero paradas e distâncias altas dentro do domínio.",\n            "Manter em auditoria as viagens sem cadastro de estação.",\n        ],\n        "unidades": UNIDADES,\n    }\n    return uniao.drop(columns="_merge"), auditoria\n\n\ndef salvar_json(caminho, dados):\n    caminho.write_text(\n        json.dumps(dados, ensure_ascii=False, indent=2, sort_keys=True) + "\\n",\n        encoding="utf-8",\n    )\n', encoding='utf-8')
_refresh_imports()


In [ ]:
from pathlib import Path
from auditoria_viagens import auditar, ler_estacoes, ler_viagens

brutas = ler_viagens(Path("dados/viagens.csv"))
estacoes = ler_estacoes(Path("dados/estacoes.csv"))
validas, auditoria = auditar(brutas, estacoes)

from auditoria_viagens import salvar_json
salvar_json(Path("auditoria.json"), auditoria)
assert set(validas["id"]) == {f"V{i:02}" for i in range(1, 11)}
print(auditoria["linhas_brutas"], auditoria["linhas_validas"])
print(auditoria["duplicatas_excedentes"])
print(auditoria["rejeitadas"])
print(auditoria["km_ausentes_validas"], auditoria["viagens_sem_cadastro"])


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
12 10
{'V04': 1}
{'V11': ['km_planejados negativo']}
['V03', 'V08'] ['V08']
```

V11 falha pelo domínio, V04 aparece uma vez na saída e V08 permanece mesmo com estação desconhecida. V07 tem valor alto porém válido. Um conflito de ID deve produzir erro em vez de apagar uma versão. A solução inclui as regras no arquivo auditoria_viagens.py.
